In [ ]:
# -*- coding: utf-8 -*-
"""
天生智慧/数据库数据 批量打细分标签脚本 (DB适配版) Final NO-STRICT VERSION v6.6-db + SWITCHES v6.6-db-final
--------------------------------------------------------------------------------
⚠️ 本版本严格基于你提供的 v6.6-db 原脚本做“增量改动”，不删除/不缺失任何省份规则与裁决逻辑。

新增功能（按你的要求）：
1) ✅ 开关A：是否输出/显示“专业全称”列（仅控制写回，不影响匹配）
2) ✅ 开关B：是否将细分标签聚合到一列（';'分隔），并且在该模式下不输出“原文字段”列（也不生成各标签单列）
3) ✅ 在原合成“专业全称”基础上新增 5 个备用字段列（源自原表；可为空；列不存在/为空均不报错、不使用）

本次新增规则修正（不删除原规则，仅追加裁决）：
4) ✅ 避免“中外合作办学专业除外/不含/不包括”等否定语境误打“中外合作”标签
5) ✅ 若同时命中「地方公费师范生」+「地方优师专项」=> 仅保留「地方优师专项」（原文字段同步）
6) ✅ 若同时命中「国家公费师范生」+「国家优师专项」=> 仅保留「国家优师专项」（原文字段同步）

使用说明：
- 修改 RULE_PATH / INPUT_PATH / OUTPUT_DIR 为你的本地路径
- DB_COL_SPARES 可改成你真实表头；即便不存在也不会报错
- 输出：在 OUTPUT_DIR 下生成：{原文件名}_demo.xlsx
"""

import os
import re
from openpyxl import load_workbook

# =========================
# 配置区（你自己改路径）
# =========================
RULE_PATH = r"规则表/各省份升学路径细分.xlsx"

# ✅ 支持：单个文件 或 目录
INPUT_PATH = r"原文件表/内部数据库数据原文件/24年和23年数据/浙江"
OUTPUT_DIR = r"输出结果/内部数据库数据输出结果/浙江"

# =========================
# ✅ 新增：两个开关
# =========================
# 开关A：是否写回/显示“专业全称”列（只影响输出列，不影响内部用于匹配的合成文本）
SW_SHOW_MAJOR_FULLNAME = False

# 开关B：是否聚合细分标签到一列（';'分隔）
# - True：只输出聚合列；不输出“原文字段”列；不生成各标签单独列
# - False：保持原输出（多标签列 + 原文字段列）
SW_AGGREGATE_LABELS = True
OUT_COL_AGG_LABELS = "专业标签"  # 聚合输出列名（开关B=True时生效）

# 规则表列名
RULE_COL_PROVINCE = "省份"
RULE_COL_LABEL = "细分标签"
RULE_COL_KEYS = "提取字段"

# DB 数据列名（你这份文件里是这些）
DB_COL_PROVINCE = "生源地"
DB_COL_MAJOR_NAME = "专业名称"
DB_COL_MAJOR_DIR = "专业方向"
DB_COL_MAJOR_REMARK = "专业备注"
DB_COL_SCHOOL = "院校名称"

# ✅ 新增：5个备用字段列（“必定源自原表”但可能某些文件没有该列；没有也不报错，视为全空不用）
# 你可以把下面五个名字改成你真实表头；不改也能跑（不存在即忽略）
DB_COL_SPARES = [
    "专业组备注",
    "",
    "批次",
    "",
    "",
]

# 写回列
DB_COL_MAJOR_FULLNAME = "专业全称"   # ✅ 会自动创建/覆盖（由开关A控制是否写回）
OUT_COL_HITS = "原文字段"           # ✅ 命中的 key 串（开关B=True时不输出）

# 七所部属师范补标用
NATIONAL_GONGFEI_SCHOOLS = {
    "华东师范大学",
    "华中师范大学",
    "北京师范大学(珠海校区)",
    "北京师范大学",
    "西南大学",
    "陕西师范大学",
    "东北师范大学",
}

# 新疆特殊
RE_YOUSHI = re.compile(r"优师专项")
RE_DINGXIANG_JOB = re.compile(r"定向(.+?)就业")

NEG_PREFIX = {"非", "不"}

NEG_CONTEXT_GONGFEI = re.compile(
    r"(不招收公费|不招公费|不接收|不收|禁止|不得|不允许|非|不包含)\s*[：:（(【\[]?\s*(公费师范生|公费师范)"
)

# 省份级排除规则
EXCLUDE_LABEL_BY_PATTERN = {
    "海南": [
        ("预科班", re.compile(r"预科班[（(]\s*民族班\s*[)）]")),
    ]
}

# 冲突裁决（keep > drop）
CONFLICT_KEEP_RULES = [
    ("定向培养军士生", "定向就业"),
    ("定向医学生", "定向就业"),
]

# =========================
# 工具函数
# =========================
def header_to_col(ws):
    m = {}
    for c in range(1, ws.max_column + 1):
        v = ws.cell(row=1, column=c).value
        if v is None:
            continue
        m[str(v).strip()] = c
    return m

def ensure_col(ws, header_map, name):
    if name in header_map:
        return header_map[name]
    new_col = ws.max_column + 1
    ws.cell(row=1, column=new_col).value = name
    header_map[name] = new_col
    return new_col

def get_str(v):
    return "" if v is None else str(v).strip()


def norm_prov_name(prov: str) -> str:
    """用于省份特殊规则判断：去掉常见后缀（省/市/自治区），不影响规则表查找。"""
    p = get_str(prov)
    for suf in ("省", "市"):
        if p.endswith(suf):
            p = p[:-1]
    # 自治区：常见写法（兼容“内蒙古自治区”等）
    p = p.replace("壮族自治区", "").replace("回族自治区", "").replace("维吾尔自治区", "").replace("自治区", "")
    return p.strip()

def split_multikey_cell(cell_text: str):
    if not cell_text:
        return []
    s = str(cell_text).strip()
    if not s:
        return []
    keys = []
    for line in re.split(r"[\r\n]+", s):
        line = line.strip()
        if not line:
            continue
        keys.extend([p.strip() for p in line.split("/") if p.strip()])
    return keys

def parse_key_expr(key: str):
    key = key.strip()
    if not key:
        return None
    if "+" in key:
        parts = [p.strip() for p in key.split("+") if p.strip()]
        return ("AND", parts, key) if parts else None
    return ("SINGLE", [key], key)

def token_sort_key(item):
    return (len(item["raw"]), sum(len(p) for p in item["parts"]))

def is_wildcard_key(key: str):
    return ("XXX" in key) or ("xxx" in key) or ("X" in key) or ("x" in key)

def compile_wildcard_regex(key: str):
    esc = re.escape(key)
    esc = esc.replace(re.escape("XXX"), r"(.+?)")
    esc = esc.replace(re.escape("xxx"), r"(.+?)")
    esc = esc.replace(re.escape("X"), r"(.+?)")
    esc = esc.replace(re.escape("x"), r"(.+?)")
    return re.compile(esc)

def has_non_negated_literal(text: str, literal: str) -> bool:
    if "公费" not in literal:
        return literal in text

    start = 0
    while True:
        idx = text.find(literal, start)
        if idx == -1:
            return False
        if idx == 0:
            return True
        if text[idx - 1] not in NEG_PREFIX:
            return True
        start = idx + 1

def match_part(text: str, part: str) -> bool:
    return has_non_negated_literal(text, part)

def match_expr(text: str, kind: str, parts: list[str], raw_key: str):
    """
    ✅ no strict fallback：
    - SINGLE: part in text
    - AND: all part in text
    - wildcard: regex
    """
    if is_wildcard_key(raw_key):
        rg = compile_wildcard_regex(raw_key)
        m = rg.search(text)
        if not m:
            return False, None
        return True, m.group(0)

    if kind == "SINGLE":
        ok = match_part(text, parts[0])
        return ok, (raw_key if ok else None)

    if kind == "AND":
        for p in parts:
            if not match_part(text, p):
                return False, None
        return True, raw_key

    return False, None

def match_label_by_rules(prov_val: str, target_label: str, remark: str, prov_rules: dict):
    entries = prov_rules.get(prov_val)
    if not entries:
        return False, None

    for entry in entries:
        if entry["label"] != target_label:
            continue

        for it in entry["items"]:
            ok, hit_str = match_expr(
                remark,
                it["kind"],
                it["parts"],
                it["raw"],
            )
            if ok:
                return True, hit_str
        return False, None

    return False, None

# =========================
# 文本归一化 + 专业全称合成（DB）
# =========================
_symbol_map = {
    "（": "(", "）": ")",
    "【": "[", "】": "]",
    "，": ",", "、": ",",
    "；": ";", "：": ":",
    "－": "-", "—": "-", "–": "-",
    "＋": "+", "／": "/", "．": ".",
    "\u200b": "", "\xa0": "", "\u3000": "",
    "“": "", "”": "", "‘": "", "’": "",
    "。": "", "！": "", "？": "",
    "　": " ", "\t": " ",
}
_symbol_re = re.compile("|".join(map(re.escape, _symbol_map.keys())))

def unify_symbols(s: str) -> str:
    if not s:
        return ""
    return _symbol_re.sub(lambda m: _symbol_map[m.group(0)], str(s))

def apply_synonyms(s: str) -> str:
    if not s:
        return ""
    s = str(s)

    # 含： -> 包含专业:
    s = re.sub(r"含[：:]", "包含专业:", s)
    s = re.sub(r"(包)*包含专业(专业)*", "包含专业", s)

    s = s.replace("国家专项计划组", "国家专项计划")
    s = s.replace("中外合作办学组", "中外合作办学")
    s = s.replace("中外合作办学或合作项目组", "中外合作办学")
    s = s.replace("中外合作办学或合作项目", "中外合作办学")
    s = s.replace("合作项目组", "合作项目")
    return s

def remove_all_spaces(s: str) -> str:
    if not s:
        return ""
    s = str(s)
    s = s.replace("\u200b", "").replace("\xa0", "").replace("\u3000", "")
    return re.sub(r"\s+", "", s)

def _norm_piece(s: str) -> str:
    """用于去重判断：同义词+符号统一+去空白"""
    s = apply_synonyms(s)
    s = unify_symbols(s)
    s = remove_all_spaces(s)
    return s

def build_major_fullname_db(major_name: str, major_dir: str, major_remark: str, spare_values: list[str] | None = None) -> str:
    """
    生成“专业全称”（可用于规则匹配，也写回 Excel）：
    - 原三字段：专业名称 + 批次 + 专业备注
    - ✅ 新增：最多5个备用字段（来自原表，可能为空；为空/不存在则跳过，不报错）
    - 去重规则：先做同义词/符号统一/去空白，再判断“是否已包含”，避免重复拼接
    """
    spare_values = spare_values or []

    # 原三字段 + 备用字段
    pieces = [major_name, major_dir, major_remark] + list(spare_values)

    parts_show = []
    merged_norm = ""

    for piece in pieces:
        p = _norm_piece(piece)
        if not p:
            continue
        if p in merged_norm:
            continue
        parts_show.append(p)
        merged_norm += p

    return "".join(parts_show)

# =========================
# build_rules：提取字段为空 → keys=[label]（普通匹配）
# =========================
def build_rules(rule_ws):
    h = header_to_col(rule_ws)
    for need in [RULE_COL_PROVINCE, RULE_COL_LABEL, RULE_COL_KEYS]:
        if need not in h:
            raise ValueError(f"规则表缺少列：{need}")

    col_p = h[RULE_COL_PROVINCE]
    col_l = h[RULE_COL_LABEL]
    col_k = h[RULE_COL_KEYS]

    prov_map = {}
    label_keycount = {}

    for r in range(2, rule_ws.max_row + 1):
        prov = get_str(rule_ws.cell(r, col_p).value)
        label = get_str(rule_ws.cell(r, col_l).value)
        keys_cell = get_str(rule_ws.cell(r, col_k).value)

        if not prov or not label:
            continue

        keys = split_multikey_cell(keys_cell)

        # ✅ 不再 strict：为空就兜底 label
        if not keys:
            keys = [label]

        seen = set()
        keys2 = []
        for k in keys:
            k = k.strip()
            if not k:
                continue
            if k not in seen:
                keys2.append(k)
                seen.add(k)

        label_keycount[label] = max(label_keycount.get(label, 0), len(keys2))

        items = []
        for k in keys2:
            parsed = parse_key_expr(k)
            if not parsed:
                continue
            kind, parts, raw = parsed
            items.append({
                "kind": kind,
                "parts": parts,
                "raw": raw,
            })

        items.sort(key=token_sort_key, reverse=True)
        prov_map.setdefault(prov, []).append({"label": label, "items": items})

    all_labels = sorted(label_keycount.keys())
    return prov_map, all_labels, label_keycount

# =========================
# 处理单个文件（DB）
# =========================
def process_one_file(filepath, prov_rules, all_labels, label_keycount):
    wb = load_workbook(filepath)
    ws = wb.active
    h = header_to_col(ws)

    # 必要列检查
    for need in [DB_COL_PROVINCE, DB_COL_MAJOR_NAME, DB_COL_MAJOR_DIR, DB_COL_MAJOR_REMARK]:
        if need not in h:
            raise ValueError(f"{os.path.basename(filepath)} 缺少列：{need}")

    has_school = DB_COL_SCHOOL in h
    col_school = h.get(DB_COL_SCHOOL)

    col_prov = h[DB_COL_PROVINCE]
    col_name = h[DB_COL_MAJOR_NAME]
    col_dir = h[DB_COL_MAJOR_DIR]
    col_rem = h[DB_COL_MAJOR_REMARK]

    # ✅ 备用字段列（可能不存在；不存在则视为全空，不报错）
    spare_cols = [h.get(colname) for colname in DB_COL_SPARES]

    # ✅ 写回：专业全称列（由开关A控制是否创建/写回）
    col_full = None
    if SW_SHOW_MAJOR_FULLNAME:
        col_full = ensure_col(ws, h, DB_COL_MAJOR_FULLNAME)

    matched_any_label = {lbl: False for lbl in all_labels}
    row_cache = [None] * (ws.max_row + 1)

    # 先逐行计算专业全称 + 逐行匹配
    for r in range(2, ws.max_row + 1):
        prov_val = get_str(ws.cell(r, col_prov).value)
        major_name = get_str(ws.cell(r, col_name).value)
        major_dir = get_str(ws.cell(r, col_dir).value)
        major_remark = get_str(ws.cell(r, col_rem).value)
        school = get_str(ws.cell(r, col_school).value) if has_school else ""

        # ✅ 新增：读取5个备用字段（列不存在/值为空均OK）
        spare_vals = []
        for cidx in spare_cols:
            if cidx is None:
                spare_vals.append("")
            else:
                spare_vals.append(get_str(ws.cell(r, cidx).value))

        # ✅ 合成专业全称（用于匹配；写回由开关A决定）
        major_full = build_major_fullname_db(major_name, major_dir, major_remark, spare_vals)

        # 开关A：是否写回“专业全称”
        if SW_SHOW_MAJOR_FULLNAME and col_full is not None:
            ws.cell(r, column=col_full).value = major_full

        if not prov_val or not major_full:
            row_cache[r] = ([], [])
            continue

        entries = prov_rules.get(prov_val)
        if not entries:
            row_cache[r] = ([], [])
            continue

        matched_labels = []
        label_seen = set()
        label_to_hitkey = {}

        remark = major_full  # ✅ 用合成后的“专业全称”去匹配

        # 新疆优师专项特殊规则
        if norm_prov_name(prov_val) == "新疆":
            special_done_youshi = False
            if RE_YOUSHI.search(remark) and RE_DINGXIANG_JOB.search(remark):
                matched_labels.append("地方优师专项")
                label_seen.add("地方优师专项")
                matched_any_label["地方优师专项"] = True
                m = RE_DINGXIANG_JOB.search(remark)
                label_to_hitkey["地方优师专项"] = f"定向{m.group(1)}就业+优师专项"
                special_done_youshi = True

            if (not special_done_youshi) and RE_YOUSHI.search(remark):
                matched_labels.append("国家优师专项")
                label_seen.add("国家优师专项")
                matched_any_label["国家优师专项"] = True
                label_to_hitkey["国家优师专项"] = "优师专项"

        # 通用细类匹配
        for entry in entries:
            label = entry["label"]

            if prov_val == "新疆" and label in {"国家优师专项", "地方优师专项"}:
                continue
            if label in label_seen:
                continue

            first_hit = None
            for it in entry["items"]:
                ok, hit_str = match_expr(
                    remark,
                    it["kind"],
                    it["parts"],
                    it["raw"],
                )
                if ok:
                    first_hit = hit_str
                    break

            if first_hit:
                matched_labels.append(label)
                label_seen.add(label)
                matched_any_label[label] = True
                label_to_hitkey[label] = first_hit

        # 省份级排除规则
        rules = EXCLUDE_LABEL_BY_PATTERN.get(prov_val, [])
        if rules and matched_labels:
            remove_set = set()
            for (bad_label, rg) in rules:
                if rg.search(remark):
                    remove_set.add(bad_label)

            if remove_set:
                matched_labels = [x for x in matched_labels if x not in remove_set]
                for x in list(label_seen):
                    if x in remove_set:
                        label_seen.discard(x)
                        label_to_hitkey.pop(x, None)

        # =========================
        # ✅ 新增修正：中外合作“除外/不含/不包括”否定语境，禁止打“中外合作”标签
        # 说明：如“(中外合作办学专业除外)”包含“中外合作”但语义是否定，需要剔除
        # =========================
        if "中外合作" in label_seen:
            if re.search(r"(?:中外合作(?:办学)?.*?(?:除外|不含|不包括))|(?:(?:不含|除|不包括|非|不是|排除).*?中外合作(?:办学)?)", remark):
                matched_labels = [l for l in matched_labels if l != "中外合作"]
                label_seen.discard("中外合作")
                label_to_hitkey.pop("中外合作", None)

        # ✅ 内蒙古覆盖：边防军人子女预科班 > 预科班
        if norm_prov_name(prov_val) == "内蒙古":
            keep_label = "边防军人子女预科班"
            drop_label = "预科班"
            if keep_label in label_seen and drop_label in label_seen:
                matched_labels = [l for l in matched_labels if l != drop_label]
                label_seen.discard(drop_label)
                label_to_hitkey.pop(drop_label, None)

        # 辽宁纠错：地方专项覆盖高校专项
        if norm_prov_name(prov_val) == "辽宁":
            if "地方专项" in label_seen and "高校专项" in label_seen:
                hit_local = label_to_hitkey.get("地方专项", "")
                if "高校专项" in hit_local:
                    matched_labels = [l for l in matched_labels if l != "高校专项"]
                    label_seen.discard("高校专项")
                    label_to_hitkey.pop("高校专项", None)

        # 福建公费师范裁决
        if norm_prov_name(prov_val) == "福建":
            has_public = has_non_negated_literal(remark, "公费师范")
            has_national = "国家公费师范生" in label_seen
            has_compound = "复合型公费师范" in label_seen
            has_provincial = "省属公费师范生" in label_seen

            if has_national or has_compound:
                if has_provincial:
                    matched_labels = [l for l in matched_labels if l != "省属公费师范生"]
                    label_seen.discard("省属公费师范生")
                    label_to_hitkey.pop("省属公费师范生", None)

            elif has_public and not has_provincial:
                matched_labels.append("省属公费师范生")
                label_seen.add("省属公费师范生")
                matched_any_label["省属公费师范生"] = True
                label_to_hitkey["省属公费师范生"] = "公费师范"

        # 七所部属师范院校：国家公费师范生补标
        if has_school:
            school_norm = school.replace("（", "(").replace("）", ")").strip()
            if school_norm in NATIONAL_GONGFEI_SCHOOLS:
                if "国家公费师范生" not in label_seen:
                    ok, hit_key = match_label_by_rules(prov_val, "国家公费师范生", remark, prov_rules)
                    if not ok:
                        has_gongfei = ("公费师范生" in remark) or ("公费师范" in remark)
                        neg_ctx = bool(NEG_CONTEXT_GONGFEI.search(remark))
                        if has_gongfei and not neg_ctx:
                            ok = True
                            hit_key = "公费师范生" if "公费师范生" in remark else "公费师范"
                    if ok and NEG_CONTEXT_GONGFEI.search(remark):
                        ok = False
                    if ok:
                        matched_labels.append("国家公费师范生")
                        label_seen.add("国家公费师范生")
                        matched_any_label["国家公费师范生"] = True
                        label_to_hitkey["国家公费师范生"] = hit_key


        # 山西规则：国家公费师范生覆盖地方公费师范生（需放在“七所部属师范补标”之后再次裁决）
        # 说明：西南大学/陕西师大等学校的“国家公费师范生”可能由补标逻辑后置添加，
        #      若仅在补标前裁决，会出现最终同时保留两个标签的情况。
        if norm_prov_name(prov_val) == "山西":
            keep_label = "国家公费师范生"
            drop_label = "地方公费师范生"
            if keep_label in label_seen and drop_label in label_seen:
                matched_labels = [l for l in matched_labels if l != drop_label]
                label_seen.discard(drop_label)
                label_to_hitkey.pop(drop_label, None)

        # 冲突裁决（定向医/军士 > 定向就业）
        for keep_label, drop_label in CONFLICT_KEEP_RULES:
            if keep_label in label_seen and drop_label in label_seen:
                matched_labels = [l for l in matched_labels if l != drop_label]
                label_seen.discard(drop_label)
                label_to_hitkey.pop(drop_label, None)

        # ✅✅✅ 新增冲突裁决：公费类/军校类 禁止打“定向就业”
        # 规则：
        # - 若已命中以下任一“公费”标签，则不得再命中“定向就业”
        # - 若院校名称包含“军”，也不得命中“定向就业”
        GONGFEI_BLOCK_LABELS = {
            "公费农科生",
            "国家公费师范生",
            "地方公费师范生",
            "省属公费师范生",
            "省属公费农科生",
        }

        if "定向就业" in label_seen:
            has_gongfei_block = any(lbl in label_seen for lbl in GONGFEI_BLOCK_LABELS)
            has_military_school = ("军" in school) if has_school else False

            if has_gongfei_block or has_military_school:
                matched_labels = [l for l in matched_labels if l != "定向就业"]
                label_seen.discard("定向就业")
                label_to_hitkey.pop("定向就业", None)

        # 优师类优先裁决（含优师 > 定向就业）
        if "定向就业" in label_seen:
            youshi_labels = [lbl for lbl in label_seen if "优师" in lbl]
            if youshi_labels:
                matched_labels = [l for l in matched_labels if l != "定向就业"]
                label_seen.discard("定向就业")
                label_to_hitkey.pop("定向就业", None)

        # =========================
        # ✅ 新增裁决：优师专项覆盖公费师范（且原文字段同步）
        # - 地方优师专项 > 地方公费师范生
        # - 国家优师专项 > 国家公费师范生
        # =========================
        if "地方优师专项" in label_seen and "地方公费师范生" in label_seen:
            matched_labels = [l for l in matched_labels if l != "地方公费师范生"]
            label_seen.discard("地方公费师范生")
            label_to_hitkey.pop("地方公费师范生", None)

        if "国家优师专项" in label_seen and "国家公费师范生" in label_seen:
            matched_labels = [l for l in matched_labels if l != "国家公费师范生"]
            label_seen.discard("国家公费师范生")
            label_to_hitkey.pop("国家公费师范生", None)

        hit_keys = [label_to_hitkey[lbl] for lbl in matched_labels if lbl in label_to_hitkey]
        row_cache[r] = (matched_labels, hit_keys)

    # =========================
    # ✅ 新增：聚合输出模式（开关B）
    # - 聚合到一列；多标签用 ';'
    # - 不输出“原文字段”列
    # - 不生成各标签单独列
    # =========================
    if SW_AGGREGATE_LABELS:
        agg_col = ensure_col(ws, h, OUT_COL_AGG_LABELS)
        for r in range(2, ws.max_row + 1):
            matched_labels, _hit_keys = row_cache[r]
            ws.cell(r, column=agg_col).value = ";".join(matched_labels) if matched_labels else None
        return wb

    # =========================
    # 原输出逻辑：只输出“至少命中过一次”的标签列（减少空列） + 原文字段
    # =========================
    matched_labels_all = [lbl for lbl in all_labels if matched_any_label.get(lbl, False)]
    sorted_labels = sorted(matched_labels_all, key=lambda lbl: (-label_keycount.get(lbl, 0), lbl))

    label_to_col = {}
    for lbl in sorted_labels:
        label_to_col[lbl] = ensure_col(ws, h, lbl)
    hit_col = ensure_col(ws, h, OUT_COL_HITS)

    # 写标签值
    for r in range(2, ws.max_row + 1):
        matched_labels, hit_keys = row_cache[r]
        for lbl in sorted_labels:
            ws.cell(r, column=label_to_col[lbl]).value = None
        ws.cell(r, column=hit_col).value = None

        for lbl in matched_labels:
            if lbl in label_to_col:
                ws.cell(r, column=label_to_col[lbl]).value = lbl

        if hit_keys:
            ws.cell(r, column=hit_col).value = " ".join(hit_keys)

    return wb

# =========================
# 主入口
# =========================
def iter_input_files(input_path: str):
    if os.path.isfile(input_path):
        return [input_path]
    if os.path.isdir(input_path):
        out = []
        for fn in os.listdir(input_path):
            if fn.startswith("~$"):
                continue
            if not (fn.lower().endswith(".xlsx") or fn.lower().endswith(".xlsm")):
                continue
            out.append(os.path.join(input_path, fn))
        return sorted(out)
    raise ValueError(f"INPUT_PATH 不存在：{input_path}")

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    rule_wb = load_workbook(RULE_PATH)
    rule_ws = rule_wb.active
    prov_rules, all_labels, label_keycount = build_rules(rule_ws)

    files = iter_input_files(INPUT_PATH)
    if not files:
        raise ValueError("未找到可处理的输入文件")

    for src in files:
        fn = os.path.basename(src)
        base, _ = os.path.splitext(fn)
        dst = os.path.join(OUTPUT_DIR, f"{base}_fix.xlsx")

        try:
            wb_out = process_one_file(src, prov_rules, all_labels, label_keycount)
            wb_out.save(dst)
            print(f"[OK] {fn} -> {os.path.basename(dst)}")
        except Exception as e:
            print(f"[FAIL] {fn}: {e}")

if __name__ == "__main__":
    main()